In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import json
import socket
import re

from utils_mitgcm import *
from utils_eof import *

In [ ]:
lake = 'neuchatel'
model = f'{lake}_2025'

In [ ]:
with open('../../../config.json', 'r') as file:
    config = json.load(file)[socket.gethostname()][model]
base_folder_path = os.path.dirname(config['datapath'])

In [ ]:
base_folder = os.path.join(base_folder_path, "eof", "synthetic_events")

# Fetch KE projected for each EOF

In [ ]:
# Extract and list the dates of the extracted EOF patterns
_all_eof_subfolders = [
    name for name in os.listdir(base_folder)
    if os.path.isdir(os.path.join(base_folder, name))
]

_pat = re.compile(r"(\d{4}-\d{2}-\d{2})$")

eof_folders = [name for name in _all_eof_subfolders if _pat.match(name)]
eof_dates_str = [m.group(1) for name in eof_folders for m in [_pat.match(name)]]

In [ ]:
eof_dates_str

In [ ]:
ke_list = []
for eof_date in eof_dates_str:
    ke_temp = pd.read_csv(os.path.join(base_folder, eof_date, "ke_projected_eof.csv"))
    ke_temp['time'] = pd.to_datetime(ke_temp['time'])
    ke_temp = ke_temp.set_index('time')
    ke_list.append(ke_temp)

In [ ]:
plt.figure(figsize=(25, 5))
plt.ylabel('Kinetic energy [MJ]')
for ke_timeserie in ke_list:
    ke_timeserie['kinetic_energy_[MJ]'].plot()

In [ ]:
# Merge all KE time series and average values that fall on the same calendar date
ke_merged = pd.concat(ke_list, axis=0)

ke_merged = (
    ke_merged.assign(date=ke_merged.index.normalize())
    .groupby("time", sort=True)
    .mean(numeric_only=True)
)

ke_merged['kinetic_energy_[MJ]'].plot()


In [ ]:
ke_merged['kinetic_energy_[MJ]'].to_csv(os.path.join(base_folder, "ke_projected_eof.csv"))